# BigNeuron Benchmark — Evaluation

Compares three methods against BigNeuron gold standard SWCs.

| Method | Description |
|--------|-------------|
| **ours** | step1→step2→step3 (orient_field streamlines, GWDT burn) |
| **vaa3d** | Vaa3D app2 (APP2 algorithm) |
| **rivuletpy** | Original rivuletpy CLI |

**Metrics (via pyneval)**
- `ssd` — Spatial Structure Distance (lower = better)
- `length` — overlapping branch length ratio
- `critical_node` — branching/terminal node accuracy

## Setup
```bash
pip install pyneval pandas matplotlib seaborn
```

In [ ]:
import subprocess, sys
# Uncomment to install:
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyneval', 'pandas', 'matplotlib', 'seaborn'], check=True)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import subprocess, json, warnings
warnings.filterwarnings('ignore')

ROOT        = Path('..').resolve()
GOLD_DIR    = ROOT / 'data' / 'gold_standard'
RESULTS_DIR = ROOT / 'results'
METHODS     = ['ours', 'vaa3d', 'rivuletpy']

print(f'Root      : {ROOT}')
print(f'Gold SWCs : {len(list(GOLD_DIR.glob("*.swc")))}')
for m in METHODS:
    n = len(list((RESULTS_DIR / m).glob('*.swc')))
    print(f'  {m:12s}: {n} result SWCs')

## Compute Metrics

In [ ]:
def pyneval_score(test_swc: Path, gold_swc: Path, metric: str = 'ssd') -> float:
    """Run pyneval CLI and return the scalar score."""
    result = subprocess.run(
        ['pyneval', '--test', str(test_swc), '--gold', str(gold_swc), '--metric', metric],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        return float('nan')
    # pyneval prints: "metric_name : value"
    for line in result.stdout.splitlines():
        if ':' in line:
            try:
                return float(line.split(':')[-1].strip())
            except ValueError:
                pass
    return float('nan')


METRICS = ['ssd', 'length', 'critical_node']
rows = []

gold_swcs = sorted(GOLD_DIR.glob('*.swc'))
print(f'Evaluating {len(gold_swcs)} samples x {len(METHODS)} methods x {len(METRICS)} metrics ...')

for gold_swc in gold_swcs:
    sample = gold_swc.stem
    for method in METHODS:
        test_swc = RESULTS_DIR / method / f'{sample}.swc'
        if not test_swc.exists():
            continue
        row = {'sample': sample, 'method': method}
        for metric in METRICS:
            row[metric] = pyneval_score(test_swc, gold_swc, metric)
        rows.append(row)
        print(f'  {method:12s}  {sample}  ssd={row["ssd"]:.4f}')

df = pd.DataFrame(rows)
df.to_csv('results_raw.csv', index=False)
print(f'\nSaved results_raw.csv  ({len(df)} rows)')
df.head()

## Summary Table

In [ ]:
summary = df.groupby('method')[METRICS].agg(['mean', 'std']).round(4)
print(summary.to_string())
summary

## Plots

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 5))
palette = {'ours': '#4c9be8', 'vaa3d': '#e87b4c', 'rivuletpy': '#6ec26e'}

for ax, metric in zip(axes, METRICS):
    sns.boxplot(data=df, x='method', y=metric, palette=palette, ax=ax)
    sns.stripplot(data=df, x='method', y=metric, color='white', size=3, alpha=0.6, ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('')

plt.suptitle('BigNeuron Benchmark', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved benchmark_results.png')

## Per-sample SSD comparison (ours vs baselines)

In [ ]:
pivot = df.pivot(index='sample', columns='method', values='ssd')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, baseline in zip(axes, ['vaa3d', 'rivuletpy']):
    if 'ours' not in pivot.columns or baseline not in pivot.columns:
        ax.set_visible(False)
        continue
    sub = pivot[['ours', baseline]].dropna()
    lo  = min(sub.min().min(), 0)
    hi  = sub.max().max()
    ax.scatter(sub[baseline], sub['ours'], alpha=0.7)
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1)
    ax.set_xlabel(f'{baseline} SSD')
    ax.set_ylabel('ours SSD')
    ax.set_title(f'ours vs {baseline}  (below diagonal = ours better)')
    # annotate win rate
    n_better = int((sub['ours'] < sub[baseline]).sum())
    ax.text(0.05, 0.95, f'ours better: {n_better}/{len(sub)}',
            transform=ax.transAxes, va='top', color='navy')

plt.tight_layout()
plt.savefig('benchmark_scatter.png', dpi=150, bbox_inches='tight')
plt.show()